In [6]:
from pathlib import Path
import importlib
import sys

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import utils.common as common_module
import utils.data as data_module
import utils.adaptive_routing as adaptive_module
import utils.losses as losses_module

common_module = importlib.reload(common_module)
data_module = importlib.reload(data_module)
adaptive_module = importlib.reload(adaptive_module)
losses_module = importlib.reload(losses_module)

BCEDiceLoss = losses_module.BCEDiceLoss
DEFAULT_CUBE_SIZES = data_module.DEFAULT_CUBE_SIZES
BereaPatchDataset = data_module.BereaPatchDataset
CubeSizeBatchSampler = data_module.CubeSizeBatchSampler
TopologyAdaptiveRoutedUNet3D = adaptive_module.TopologyAdaptiveRoutedUNet3D
auxiliary_physics_loss = losses_module.auxiliary_physics_loss
dice_score_from_logits = losses_module.dice_score_from_logits
embedding_consistency_loss = losses_module.embedding_consistency_loss
topology_prediction_loss = losses_module.topology_prediction_loss
from utils.training import EarlyStopping, MetricTracker
from utils.topology import TOPOLOGY_FEATURE_DIM

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: f:\PycharmProjects\micro_ct device: cuda


In [7]:
TRAIN_MODE = "quick"  # quick | full
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
BATCH_SIZE_BY_CUBE_SIZE = {64: 2, 128: 1, 192: 1}
EPOCHS = 10
LR = 1e-4
AUX_WEIGHT = 0.05
CONSISTENCY_WEIGHT = 0.05
TOPOLOGY_WEIGHT = 0.01
PATIENCE = 3
MIN_DELTA = 1e-4
BASE_CHANNELS = 16
CTX_DIM = 64
PH_DIM = TOPOLOGY_FEATURE_DIM
SAMPLES_PER_GROUP = 8 if TRAIN_MODE == "quick" else None
MAX_TRAIN_BATCHES = 64 if TRAIN_MODE == "quick" else None
MAX_VAL_BATCHES = 16 if TRAIN_MODE == "quick" else None
SIZE_SAMPLING_WEIGHTS = {64: 0.50, 128: 0.35, 192: 0.15}
NUM_WORKERS = 0  # Windows/Jupyter: multiprocessing DataLoader often fails with spawn/pickle errors
PIN_MEMORY = device == "cuda"
USE_AMP = device == "cuda"
SAVE_CHECKPOINT = True
CHECKPOINT_PATH = MODEL_DIR / "segmentation_best.pth"


In [8]:
train_ds = BereaPatchDataset(
    ROOT,
    split="train",
    cube_size=CUBE_SIZES,
    use_raw_gray=False,
    balance=True,
    samples_per_group=SAMPLES_PER_GROUP,
    size_sampling_weights=SIZE_SAMPLING_WEIGHTS,
    return_topology=True,
)
val_ds = BereaPatchDataset(
    ROOT,
    split="val",
    cube_size=CUBE_SIZES,
    use_raw_gray=False,
    noise_types=["none"],
    balance=False,
    samples_per_group=SAMPLES_PER_GROUP,
    return_topology=True,
)

train_sampler = CubeSizeBatchSampler(train_ds, BATCH_SIZE_BY_CUBE_SIZE, shuffle=True, seed=42)
val_sampler = CubeSizeBatchSampler(val_ds, BATCH_SIZE_BY_CUBE_SIZE, shuffle=False, seed=42)
train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_sampler=val_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
if NUM_WORKERS != 0:
    print("warning: num_workers>0 is unreliable in Windows notebooks; prefer a .py training script")
print("mode:", TRAIN_MODE, "cube sizes:", CUBE_SIZES)
print("train:", len(train_ds), "val:", len(val_ds), "batches:", len(train_loader), len(val_loader))
print("batch sizes by cube_size:", BATCH_SIZE_BY_CUBE_SIZE, "max batches:", MAX_TRAIN_BATCHES, MAX_VAL_BATCHES)
if hasattr(train_ds, "df"):
    display(train_ds.df.groupby(["rock", "cube_size", "split"]).size().rename("samples").reset_index())


mode: quick cube sizes: [64, 128, 192]
train: 48 val: 48 batches: 36 40
batch sizes by cube_size: {64: 2, 128: 1, 192: 1} max batches: 64 16


,rock,cube_size,split,samples
0,BanderaBrown,64,train,8
1,BanderaBrown,128,train,8
2,BanderaBrown,192,train,8
3,Berea,64,train,8
4,Berea,128,train,8
5,Berea,192,train,8


In [9]:
model = TopologyAdaptiveRoutedUNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    ctx_dim=CTX_DIM,
    ph_dim=PH_DIM,
    topology_dim=PH_DIM,
).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
print("parameters:", sum(p.numel() for p in model.parameters()))


parameters: 1445714


In [10]:
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, mode="min")
history = []
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_stats = MetricTracker()
    train_bar = tqdm(train_loader, desc=f"train {epoch}/{EPOCHS}", leave=False)
    for batch_idx, batch in enumerate(train_bar):
        if MAX_TRAIN_BATCHES is not None and batch_idx >= MAX_TRAIN_BATCHES:
            break
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        ph_features = batch.get("ph_features")
        topology_target = batch.get("topology_target")
        if ph_features is not None:
            ph_features = ph_features.to(device)
        if topology_target is not None:
            topology_target = topology_target.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(x, ph_features=ph_features, return_dict=True)
            logits = out["logits"]
            seg_loss, bce_loss, dice_loss = criterion(logits, y)
            aux_loss, aux_parts = auxiliary_physics_loss(
                out,
                y,
                porosity_target=batch["porosity"].to(device),
                percolation_target=batch["percolates"].to(device),
                porosity_weight=AUX_WEIGHT,
                percolation_weight=AUX_WEIGHT,
            )
            topo_loss, topo_parts = topology_prediction_loss(
                out, topology_target, topology_weight=TOPOLOGY_WEIGHT
            )
            loss = seg_loss + aux_loss + topo_loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = x.size(0)
        with torch.no_grad():
            dice = dice_score_from_logits(logits, y)
        train_stats.update("loss", float(loss.detach().cpu()), batch_size)
        train_stats.update("seg_loss", float(seg_loss.detach().cpu()), batch_size)
        train_stats.update("aux_loss", float(aux_loss.detach().cpu()), batch_size)
        train_stats.update("topo_loss", float(topo_loss.detach().cpu()), batch_size)
        train_stats.update("bce", float(bce_loss.detach().cpu()), batch_size)
        train_stats.update("dice_loss", float(dice_loss.detach().cpu()), batch_size)
        train_stats.update("dice", float(dice.detach().cpu()), batch_size)
        train_bar.set_postfix(train_stats.postfix("loss", "bce", "dice_loss", "dice"))

    model.eval()
    val_stats = MetricTracker()
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"val {epoch}/{EPOCHS}", leave=False)
        for batch_idx, batch in enumerate(val_bar):
            if MAX_VAL_BATCHES is not None and batch_idx >= MAX_VAL_BATCHES:
                break
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            ph_features = batch.get("ph_features")
            topology_target = batch.get("topology_target")
            if ph_features is not None:
                ph_features = ph_features.to(device)
            if topology_target is not None:
                topology_target = topology_target.to(device)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(x, ph_features=ph_features, return_dict=True)
                logits = out["logits"]
                seg_loss, bce_loss, dice_loss = criterion(logits, y)
                aux_loss, aux_parts = auxiliary_physics_loss(
                    out,
                    y,
                    porosity_target=batch["porosity"].to(device),
                    percolation_target=batch["percolates"].to(device),
                    porosity_weight=AUX_WEIGHT,
                    percolation_weight=AUX_WEIGHT,
                )
                topo_loss, topo_parts = topology_prediction_loss(
                    out, topology_target, topology_weight=TOPOLOGY_WEIGHT
                )
                loss = seg_loss + aux_loss + topo_loss
            dice = dice_score_from_logits(logits, y)

            batch_size = x.size(0)
            val_stats.update("loss", float(loss.detach().cpu()), batch_size)
            val_stats.update("seg_loss", float(seg_loss.detach().cpu()), batch_size)
            val_stats.update("aux_loss", float(aux_loss.detach().cpu()), batch_size)
            val_stats.update("topo_loss", float(topo_loss.detach().cpu()), batch_size)
            val_stats.update("bce", float(bce_loss.detach().cpu()), batch_size)
            val_stats.update("dice_loss", float(dice_loss.detach().cpu()), batch_size)
            val_stats.update("dice", float(dice.detach().cpu()), batch_size)
            val_bar.set_postfix(val_stats.postfix("loss", "bce", "dice_loss", "dice"))

    train_metrics = train_stats.as_dict()
    val_metrics = val_stats.as_dict()
    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_dice": train_metrics["dice"],
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
    })

    print(
        f"epoch={epoch} "
        f"train_loss={train_metrics['loss']:.4f} train_dice={train_metrics['dice']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_dice={val_metrics['dice']:.4f}"
    )

    improved = early_stopping.step(val_metrics["loss"], epoch=epoch)
    if improved and SAVE_CHECKPOINT:
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "history": history,
            "base_channels": BASE_CHANNELS,
            "ctx_dim": CTX_DIM,
            "ph_dim": PH_DIM,
            "topology_dim": PH_DIM,
        }, CHECKPOINT_PATH)
        print("сохранено:", CHECKPOINT_PATH)
    elif improved:
        print("architecture smoke: checkpoint skipped")
    else:
        print(f"ранняя остановка: {early_stopping.bad_epochs}/{PATIENCE} эпох без улучшения val_loss")

    if early_stopping.should_stop:
        print(f"обучение остановлено на эпохе {epoch}; лучший val_loss={early_stopping.best:.4f}")
        break


C:\Users\F\AppData\Local\Temp\ipykernel_19172\3000098782.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
train 1/10:   0%|          | 0/36 [00:00<?, ?it/s]C:\Users\F\AppData\Local\Temp\ipykernel_19172\3000098782.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):
val 1/10:   0%|          | 0/40 [00:00<?, ?it/s]                                                                   C:\Users\F\AppData\Local\Temp\ipykernel_19172\3000098782.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


epoch=1 train_loss=0.3615 train_dice=0.7955 val_loss=0.2456 val_dice=0.8801
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


epoch=2 train_loss=0.2260 train_dice=0.9144 val_loss=0.2101 val_dice=0.9046
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


epoch=3 train_loss=0.1880 train_dice=0.9351 val_loss=0.1849 val_dice=0.9147
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


epoch=4 train_loss=0.1664 train_dice=0.9393 val_loss=0.1768 val_dice=0.9128
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


...

epoch=6 train_loss=0.1325 train_dice=0.9446 val_loss=0.1366 val_dice=0.9287
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


epoch=7 train_loss=0.1146 train_dice=0.9452 val_loss=0.1400 val_dice=0.9157
ранняя остановка: 1/3 эпох без улучшения val_loss


epoch=8 train_loss=0.1079 train_dice=0.9414 val_loss=0.1215 val_dice=0.9245
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


epoch=9 train_loss=0.0906 train_dice=0.9504 val_loss=0.1148 val_dice=0.9279
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth


epoch=10 train_loss=0.0845 train_dice=0.9490 val_loss=0.1128 val_dice=0.9293
сохранено: f:\PycharmProjects\micro_ct\models\segmentation_best.pth
